## Project Overview

### Business Problem
A company want to create a new movie studio but it doesn't know anything about creating it. This project investigates which types of films are currently performing best at the global box office, with the goal of translating these findings into actionable insights for the head of our company’s new movie studio. The analysis will explore how production budgets, audience preferences, and critical reception interact to shape financial outcomes, ultimately guiding investment decisions toward maximizing profitability.
### Business Understanding.
### 1. Objectives
* Return on investment: Aim is to maximize return on investment by producing films that will generate high revenue compared with production costs. Assess the relationship between production spending and box office returns to determine optimal investment strategies.
* Movie popularity: The aim will be to produce films with high audience demand to maximize viewership and increase revenues. Use IMDb and TheMovieDB data to track popularity, genre preferences, and viewer behavior.
* Reviews and Ratings: The quality of  films have an impact on the reputation of a start-up studio brand, reducing the revenues in long term. Incorporate Rotten Tomatoes scores to evaluate whether critic ratings influence box office success.
* Seasonality: We will aim to find the season that will maximize revenues. Use the Box office Mojo dataset.

### 2. Methodology
* Data integration: Combine financial performance metrics with audience and critic insights from IMDb, TheMovieDB, and Rotten Tomatoes.
* Comparative analysis: Contrast films across genres, budget ranges, and reception levels to identify patterns of profitability.
* Strategic recommendations: Translate findings into clear guidance on what types of films to prioritize, and how to align production budgets with audience demand.

### 3. Expected Outcomes
The project will deliver evidence-based recommendations for film investment. These insights will help the studio strategically allocate resources, balancing creative ambition with commercial viability to maximize returns.

### Data understanding

Different datasets were collected from different locations and the datasets are in different format and contains different data.

The datasets used in this project include:

* Box Office Mojo: This dataset provides information on domestic and international box office revenues for films. It is essential for evaluating financial performance and identifying high-grossing movies.

* IMDb: The IMDb dataset contains movie metadata, including genre classifications, runtime duration, release year, and audience ratings. These variables help analyze how film characteristics influence box office outcomes.

* Rotten Tomatoes: This dataset includes critic and audience review scores, which are used to assess how film quality and reception impact commercial success.

* TheMovieDB: TheMovieDB dataset provides additional movie attributes such as popularity scores, release dates, and audience engagement indicators. These metrics help evaluate market demand and viewer preferences.

* The Numbers: This dataset contains production budget and worldwide gross revenue information, which is critical for calculating profitability and return on investment (ROI).

### Data Preparation
The datasets used in this project come from multiple sources, each with different formats and structures. Before analysis, the data must be cleaned, standardized, and merged.



### Importing Libraries


In [56]:
# Importing necessary libraries
import pandas as pd
import sqlite3
import zipfile
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme(style="whitegrid")

### Loading Datasets

In [57]:
# Load datasets

# Box Office Mojo(domestic and foreign gross dataset)
bom = pd.read_csv("data/bom.movie_gross.csv.gz")

# The Numbers (budget dataset)
budget = pd.read_csv("data/tn.movie_budgets.csv.gz")

# We will be working with the IMDb datasets,
# so we will connect to the SQLite database and check the tables.
# Exract zipfile
with zipfile.ZipFile("data/im.db.zip", 'r') as zip_ref:
    zip_ref.extractall("data/")
# connect to database
conn = sqlite3.connect("data/im.db") 

#The MovieDB (movie opularity)
movie_db =pd.read_csv("data/tmdb.movies.csv.gz")

#Rotten Tomatoes (Audience review scores)
rotten_tomatoes = pd.read_csv("data/rt.movie_info.tsv.gz", sep='\t')





### Data Exploration & Data Cleaning

 In this section, we will inspect each dataset, and clean them separately for analysis.

### 1. Box Office Mojo Dataset

Data Exploration

In [23]:
bom.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


In [24]:
bom.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [25]:
bom.isnull().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

In [26]:
bom.duplicated().sum()

np.int64(0)

Data Cleaning

In [ ]:
# Cleaning the box office data
# We will drop rows with missing values in critical columns 
# (the 'title', 'domestic_gross', 'foreign_gross') 
# and convert the gross columns to numeric types for analysis.
bom = bom.dropna(subset=["title", "studio", "domestic_gross", "foreign_gross"])
bom.columns = bom.columns.str.lower().str.replace(" ", "_")
bom['domestic_gross'] = pd.to_numeric(bom['domestic_gross'], errors='coerce')
bom['foreign_gross'] = pd.to_numeric(bom['foreign_gross'], errors='coerce')

### 2. The Numbers Dataset

Exploration

In [40]:
budget.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 5782 non-null   int64 
 1   release_date       5782 non-null   object
 2   movie              5782 non-null   object
 3   production_budget  5782 non-null   object
 4   domestic_gross     5782 non-null   object
 5   worldwide_gross    5782 non-null   object
dtypes: int64(1), object(5)
memory usage: 271.2+ KB


In [41]:
budget.head()

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"


In [42]:
budget.isnull().sum()

id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

In [43]:
budget.duplicated().sum()

np.int64(0)

Data Cleaning

In [45]:
# Remove dollar signs and commas from production_budget, domestic_gross and worldwide_gross 
# using raw strings, then convert to float
budget['production_budget'] = budget['production_budget'].replace(r'[\$,]', '', regex=True).astype(float)
budget['domestic_gross'] = budget['domestic_gross'].replace(r'[\$,]', '', regex=True).astype(float)
budget['worldwide_gross'] = budget['worldwide_gross'].replace(r'[\$,]', '', regex=True).astype(float)
budget['release_date'] = pd.to_datetime(budget['release_date'], errors='coerce')



In [47]:
budget.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 5782 non-null   int64         
 1   release_date       5782 non-null   datetime64[ns]
 2   movie              5782 non-null   object        
 3   production_budget  5782 non-null   float64       
 4   domestic_gross     5782 non-null   float64       
 5   worldwide_gross    5782 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 271.2+ KB


### 3. Imdb Dataset

Data Exploration

In [58]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [59]:
# Load the relevant tables into pandas DataFrames
# We will load the `movie_basics` and `movie_ratings` tables from the IMDb database, 
# which contain information about movie titles, release years, genres, and ratings.
titles = pd.read_sql("SELECT * FROM movie_basics", conn)
ratings = pd.read_sql("SELECT * FROM movie_ratings", conn)

In [61]:
titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146144 entries, 0 to 146143
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   movie_id         146144 non-null  object 
 1   primary_title    146144 non-null  object 
 2   original_title   146123 non-null  object 
 3   start_year       146144 non-null  int64  
 4   runtime_minutes  114405 non-null  float64
 5   genres           140736 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 6.7+ MB


In [62]:
titles.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [64]:
titles.isnull().sum()

movie_id               0
primary_title          0
original_title        21
start_year             0
runtime_minutes    31739
genres              5408
dtype: int64

In [65]:
titles.duplicated().sum()


np.int64(0)

In [ ]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73856 entries, 0 to 73855
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   movie_id       73856 non-null  object 
 1   averagerating  73856 non-null  float64
 2   numvotes       73856 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.7+ MB


In [67]:
ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [68]:
ratings.isnull().sum()

movie_id         0
averagerating    0
numvotes         0
dtype: int64

In [69]:
ratings.duplicated().sum()

np.int64(0)

Data Cleaning

In [70]:

titles = titles.dropna(subset=["original_title", "runtime_minutes", "genres"])
